# Lojistik Regresyon Çalışma ve Uygulama Notları

Bu çalışmada, ikili sınıflandırma (binary classification) problemleri için lojistik regresyon modelinin teorik altyapısı incelenmiş; sıfırdan NumPy, Scikit-Learn ve PyTorch uygulamaları gerçekleştirilmiştir. Çalışma adımları ve elde edilen sonuçlar aşağıda sunulmuştur.

## 1. Teorik Altyapı ve Matematiksel Notlar

Lojistik regresyonda doğrusal regresyon çıktısı lojistik (sigmoid) fonksiyondan geçirilerek $0$ ile $1$ arasında olasılık değerleri elde edilmektedir.

### Doğrusal Model
$$
\eta = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p
$$
Burada $\beta_0$ sabit terimi (bias), $\beta_i$ ise değişkenlerin katsayılarıdır.

### Sigmoid Dönüşümü
Elde edilen doğrusal çıktıyı olasılığa dönüştürmek amacıyla kullanılan sigmoid fonksiyonu:
$$
p = \sigma(\eta) = \frac{1}{1 + e^{-\eta}}
$$

<img src="figures/sigmoid_curve.png" alt="Sigmoid Fonksiyonu" width="750" style="display: block; margin: auto; margin-top: 15px; margin-bottom: 15px;" />

### Logit (Log-Odds) Gösterimi
Olasılık değerini lojistik oranlara dönüştürmek için kullanılan logit fonksiyonu:
$$
\log\left(\frac{p}{1-p}\right) = \eta = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p
$$

<img src="figures/logit_curve.png" alt="Logit Fonksiyonu" width="750" style="display: block; margin: auto; margin-top: 15px; margin-bottom: 15px;" />

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# gorsellestirme ayarlarinin yapilmasi
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)

## 2. Elle Hesaplama Çalışması (Sayısal Örnek)

Algoritmanın işleyişini incelemek amacıyla, 7 öğrenciden oluşan basit bir veri seti (ders çalışma saati $x_1$ ve sınav notu $x_2$) üzerinden ilk iterasyonun hesaplama adımları gerçekleştirilmiştir.

### Örnek Veri Seti
- Çalışma Saati ($x_1$): `[2, 3, 4, 5, 6, 7, 8]`
- Sınav Notu ($x_2$): `[60, 65, 70, 75, 80, 85, 90]`
- Durum ($y$): `[0, 0, 1, 1, 1, 1, 1]` (1: Geçti, 0: Kaldı)

### Adım 1: Tasarım Matrisinin ($X$) Oluşturulması
Sabit terim (bias) için matrisin ilk sütununa 1 eklenmiştir:
$$
X = \begin{bmatrix} 1 & 2 & 60 \\ 1 & 3 & 65 \\ 1 & 4 & 70 \\ 1 & 5 & 75 \\ 1 & 6 & 80 \\ 1 & 7 & 85 \\ 1 & 8 & 90 \end{bmatrix}, \quad y = \begin{bmatrix} 0 \\ 0 \\ 1 \\ 1 \\ 1 \\ 1 \\ 1 \end{bmatrix}
$$

### Adım 2: Katsayıların Başlatılması
Başlangıç katsayıları sıfır olarak atanmıştır:
$$
\boldsymbol{\beta}^{(0)} = \begin{bmatrix} 0 \\ 0 \\ 0 \end{bmatrix}
$$

### Adım 3: Olasılık Tahminleri
Katsayılar sıfır olduğu için ilk iterasyondaki olasılıklar %50 olarak hesaplanmıştır:
$$
\eta^{(0)} = X \boldsymbol{\beta}^{(0)} = \begin{bmatrix} 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \\ 0 \end{bmatrix}
$$

$$
p^{(0)} = \sigma(\eta^{(0)}) = \begin{bmatrix} 0.5 \\ 0.5 \\ 0.5 \\ 0.5 \\ 0.5 \\ 0.5 \\ 0.5 \end{bmatrix}
$$

### Adım 4: Gradyan Hesabı ve Güncelleme
Maksimum olabilirlik fonksiyonunun gradyanı hesaplanarak ağırlıklar güncellenmiştir:
$$
\frac{\partial \ell}{\partial \boldsymbol{\beta}} = X^T(y - p)
$$

$$
y - p^{(0)} = \begin{bmatrix} -0.5 \\ -0.5 \\ 0.5 \\ 0.5 \\ 0.5 \\ 0.5 \\ 0.5 \end{bmatrix}
$$

$$
\frac{\partial \ell}{\partial \boldsymbol{\beta}} = \begin{bmatrix} 1.5 \\ 12.5 \\ 137.5 \end{bmatrix}
$$

Öğrenme oranı $\alpha = 0.01$ seçilerek birinci iterasyon sonundaki katsayılar elde edilmiştir:
$$
\boldsymbol{\beta}^{(1)} = \boldsymbol{\beta}^{(0)} + \alpha \frac{\partial \ell}{\partial \boldsymbol{\beta}} = \begin{bmatrix} 0.015 \\ 0.125 \\ 1.375 \end{bmatrix}
$$

### Katsayıların Yakınsaması
Model eğitildiğinde katsayıların şu değerlere yakınsadığı görülmüştür:
$$
\hat{\boldsymbol{\beta}} = \begin{bmatrix} -15.2 \\ 0.8 \\ 0.1 \end{bmatrix}
$$

Elde edilen nihai model denklemi:
$$
\log\left(\frac{p}{1-p}\right) = -15.2 + 0.8 \cdot \text{Hours} + 0.1 \cdot \text{Score}
$$

**Örnek Tahmin 1:** 5 saat çalışan ve sınav notu 75 olan öğrenci için olasılık:
$$
\eta = -15.2 + 0.8(5) + 0.1(75) = -3.7 \implies p = \sigma(-3.7) \approx 0.024 \quad (\%2.4)
$$

**Örnek Tahmin 2:** 7 saat çalışan ve sınav notu 85 olan öğrenci için olasılık:
$$
\eta = -15.2 + 0.8(7) + 0.1(85) = -1.1 \implies p = \sigma(-1.1) \approx 0.25 \quad (\%25)
$$

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

class ScratchLogisticRegression:
    def __init__(self, lr=0.001, epochs=5000):
        self.lr = lr
        self.epochs = epochs
        self.beta = None
        self.losses = []
        self.beta_history = []
        
    def fit(self, X, y):
        m, n = X.shape
        X_b = np.c_[np.ones((m, 1)), X]
        self.beta = np.zeros((n + 1, 1))
        y = y.reshape(-1, 1)
        
        for epoch in range(self.epochs):
            z = X_b.dot(self.beta)
            p = sigmoid(z)
            p = np.clip(p, 1e-15, 1 - 1e-15)  # sayisal kararlilik icin
            
            # kayip fonksiyonu (log loss)
            loss = -1/m * np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
            self.losses.append(loss)
            self.beta_history.append(self.beta.copy())
            
            # gradyan
            gradient = 1/m * X_b.T.dot(p - y)
            self.beta -= self.lr * gradient
            
    def predict_proba(self, X):
        m = X.shape[0]
        X_b = np.c_[np.ones((m, 1)), X]
        return sigmoid(X_b.dot(self.beta))
        
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

### Sıfırdan NumPy ile Modelleme

Modelin arka planındaki gradyan inişi mekanizmasını incelemek amacıyla lojistik regresyon sınıfı sıfırdan NumPy kullanılarak kodlanmıştır. Yukarıda sayısal hesaplamaları yapılan öğrenci veri seti bu modelle eğitilmiştir. İterasyonlar boyunca kaybın (Log-Loss) değişimi ve katsayıların yakınsama eğrileri çizdirilmiştir.

In [ ]:
X_stud = np.array([
    [2, 60],
    [3, 65],
    [4, 70],
    [5, 75],
    [6, 80],
    [7, 85],
    [8, 90]
])
y_stud = np.array([0, 0, 1, 1, 1, 1, 1])

model_scratch = ScratchLogisticRegression(lr=0.005, epochs=100000)
model_scratch.fit(X_stud, y_stud)

print("Eğitilen Katsayılar:")
print("Intercept (Beta_0):", model_scratch.beta[0, 0])
print("Çalışma Saati (Beta_1):", model_scratch.beta[1, 0])
print("Sınav Notu (Beta_2):", model_scratch.beta[2, 0])

# iterasyon grafiklerinin cizilmesi
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# kayip grafigi
ax[0].plot(model_scratch.losses, 'r-', linewidth=2)
ax[0].set_title('Log-Loss (Kayıp) Değerinin İterasyonla Değişimi', fontsize=12)
ax[0].set_xlabel('İterasyon (Epoch)')
ax[0].set_ylabel('Kayıp')

# katsayi gelisim grafigi
beta_history = np.array(model_scratch.beta_history).squeeze()
ax[1].plot(beta_history[:, 0], 'k-', label=r'$\beta_0$ (Sabit Terim)')
ax[1].plot(beta_history[:, 1], 'b-', label=r'$\beta_1$ (Çalışma Saati)')
ax[1].plot(beta_history[:, 2], 'g-', label=r'$\beta_2$ (Geçmiş Sınav Notu)')
ax[1].set_title('Katsayıların İterasyonla Değişimi', fontsize=12)
ax[1].set_xlabel('İterasyon (Epoch)')
ax[1].set_ylabel('Katsayı Değeri')
ax[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

## 3. Scikit-Learn Kütüphanesi ile Modelleme

Modeli daha büyük bir veri kümesinde test etmek amacıyla 1000 öğrenciden oluşan sentetik bir veri seti oluşturulmuştur. Veri sızıntısını önlemek ve öznitelikleri ölçeklendirmek amacıyla `StandardScaler` ve model eğitimi tek bir boru hattında (`Pipeline`) birleştirilmiştir.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# sentetik veri seti olusturulmasi (1000 ogrenci)
n_samples = 1000
study_hours = np.random.normal(5, 2, n_samples)
study_hours = np.clip(study_hours, 0, 10)

previous_scores = np.random.normal(75, 15, n_samples)
previous_scores = np.clip(previous_scores, 0, 100)

# hedef degiskenin dogrusal katsayilar uzerinden olusturulmasi
log_odds = -8 + 0.5 * study_hours + 0.05 * previous_scores
prob_pass = 1 / (1 + np.exp(-log_odds))
y = np.random.binomial(1, prob_pass, n_samples)

X = np.column_stack([study_hours, previous_scores])

# veriyi bolme (egitim ve test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# pipeline kurulumu
pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, random_state=42, max_iter=1000)
)
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

### Model Katsayıları ve Karar Sınırları Grafiklerinin Çizilmesi

Eğitilen modelin katsayıları yazdırılmış; katsayı önem bar grafiği, standartlaştırılmış veri uzayındaki karar sınırı ve olasılık tahmin yüzeyi çizdirilmiştir.

In [ ]:
scaler = pipeline.named_steps["standardscaler"]
model_lr = pipeline.named_steps["logisticregression"]

print("Model Katsayıları (Ölçeklendirilmiş):")
print("Sabit Terim (Intercept):", model_lr.intercept_[0])
print("Ders Çalışma Saati Katsayısı:", model_lr.coef_[0][0])
print("Geçmiş Not Katsayısı:", model_lr.coef_[0][1])

# katsayilarin bar grafigi ile cizilmesi
features = ['Intercept', 'Çalışma Saati', 'Geçmiş Sınav Notu']
coefficients = [model_lr.intercept_[0], model_lr.coef_[0][0], model_lr.coef_[0][1]]
colors = ['gray', 'blue', 'green']

plt.figure(figsize=(8, 5))
bars = plt.bar(features, coefficients, color=colors, edgecolor='black', width=0.6)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Lojistik Regresyon Katsayı Değerleri', fontsize=12)
plt.ylabel('Katsayı Değeri', fontsize=11)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + (0.05 if yval >= 0 else -0.15), f'{yval:.3f}', ha='center', va='bottom', fontweight='bold')
plt.show()

# olceklendirilmis uzayda karar sinirinin cizilmesi
plt.figure(figsize=(10, 6))
X_train_scaled = scaler.transform(X_train)
plt.scatter(X_train_scaled[y_train==0, 0], X_train_scaled[y_train==0, 1], color='red', label='Kaldı (0)', alpha=0.6)
plt.scatter(X_train_scaled[y_train==1, 0], X_train_scaled[y_train==1, 1], color='blue', label='Geçti (1)', alpha=0.6)

# karar siniri: w0 + w1*x1 + w2*x2 = 0 => x2 = -(w0 + w1*x1)/w2
x1_vals = np.linspace(-3, 3, 100)
w0 = model_lr.intercept_[0]
w1 = model_lr.coef_[0][0]
w2 = model_lr.coef_[0][1]
x2_vals = -(w0 + w1 * x1_vals) / w2

plt.plot(x1_vals, x2_vals, 'k--', linewidth=2, label='Karar Sınırı (p=0.5)')
plt.title('Ölçeklendirilmiş Uzayda Sınıflandırma Karar Sınırı', fontsize=12)
plt.xlabel('Ders Çalışma Saati (Standartlaştırılmış)')
plt.ylabel('Geçmiş Sınav Notu (Standartlaştırılmış)')
plt.axis([-3, 3, -3, 3])
plt.legend(fontsize=10)
plt.show()

# uc boyutlu olasilik yuzeyinin cizilmesi
from mpl_toolkits.mplot3d import Axes3D
x1_grid = np.linspace(-3, 3, 50)
x2_grid = np.linspace(-3, 3, 50)
X1_mesh, X2_mesh = np.meshgrid(x1_grid, x2_grid)

Z_logits = w0 + w1 * X1_mesh + w2 * X2_mesh
Z_probs = 1 / (1 + np.exp(-Z_logits))

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X1_mesh, X2_mesh, Z_probs, cmap='coolwarm', alpha=0.8, edgecolor='none')
ax.set_title('3 Boyutlu Olasılık Tahmin Yüzeyi', fontsize=12)
ax.set_xlabel('Ders Çalışma Saati (Standartlaştırılmış)')
ax.set_ylabel('Geçmiş Sınav Notu (Standartlaştırılmış)')
ax.set_zlabel('Geçme Olasılığı (p)')
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.show()

### Doğrusal Olmayan Sınırlar İçin Polinomsal Özellikler Denemesi

Doğrusal karar sınırlarının yetersiz kalabileceği durumlar için ikinci derece polinomsal öznitelik genişletmesi (`PolynomialFeatures`) eklenerek modelin performans farkı karşılaştırılmıştır.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly_pipeline = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    LogisticRegression(C=1.0, random_state=42, max_iter=1000)
)
poly_pipeline.fit(X_train, y_train)

poly_pred_proba = poly_pipeline.predict_proba(X_test)[:, 1]
print(f"Standart Model ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Polinomsal Model ROC AUC: {roc_auc_score(y_test, poly_pred_proba):.4f}")

## 4. Uygulama Notları ve Çıkarımlar

Model başarısını etkileyen temel parametrelere dair elde edilen çıkarımlar şu şekildedir:

*   **Hiperparametre C:** C parametresi düzenlileştirmenin (regularization) gücünü kontrol etmektedir. Küçük C değerleri ($0.01$ gibi) katsayıları sıfıra yaklaştırarak aşırı öğrenmeyi (overfitting) önlemektedir. Büyük C değerleri ($10$ veya $100$) ise modelin eğitim verisine daha yakın uyum sağlamasına izin vermektedir.
*   **Düzenlileştirme Cezası (Penalty):** L1 (Lasso) ceza yönteminin gereksiz özniteliklerin katsayılarını tamamen sıfır yaparak bir değişken seçimi sağladığı, L2 (Ridge) yönteminin ise katsayıları küçülttüğü gözlemlenmiştir.
*   **Sınıf Dengesizliği:** Hedef sınıfların dağılımı eşit olmadığında model çoğunluk sınıfına eğilim gösterebilmektedir. Bu durum `class_weight='balanced'` parametresi kullanılarak dengelenmiştir.

## 5. PyTorch Kütüphanesi ile Modelleme

Derin öğrenme çatısı olan PyTorch kullanılarak ikili lojistik regresyon modeli yapay sinir ağı katmanı biçiminde tasarlanmış ve eğitilmiştir. Elde edilen katsayılar Scikit-Learn katsayılarıyla karşılaştırılmıştır.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    class PyTorchLogisticRegression(nn.Module):
        def __init__(self):
            super(PyTorchLogisticRegression, self).__init__()
            self.linear = nn.Linear(in_features=2, out_features=1)
            
        def forward(self, x):
            return self.linear(x)

    model_bin = PyTorchLogisticRegression()
    criterion_bin = nn.BCEWithLogitsLoss()  # sigmoid fonksiyonunu da kapsayan ikili capraz entropi
    optimizer_bin = optim.SGD(model_bin.parameters(), lr=0.1)

    epochs = 2000
    for epoch in range(epochs):
        outputs = model_bin(X_tensor)
        loss = criterion_bin(outputs, y_tensor)
        
        optimizer_bin.zero_grad()
        loss.backward()
        optimizer_bin.step()
        
        if (epoch + 1) % 500 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Kayıp: {loss.item():.4f}")
            
    # katsayilari karsilastirmak icin agirliklar
    with torch.no_grad():
        w = model_bin.linear.weight[0].numpy()
        b = model_bin.linear.bias.item()
    print("\nPyTorch Katsayıları:")
    print("Intercept (b):", b)
    print("Ağırlıklar (w):", w)
except ImportError:
    print("PyTorch kütüphanesi ortamda kurulu değil.")

## 6. Sınıflar ve Araçlar Sözlüğü (Hatırlatma Notları)

Çalışmada kullanılan temel Scikit-Learn ve PyTorch bileşenlerinin ne işe yaradığına dair kısa özetler:

### Scikit-Learn Bileşenleri

*   **`StandardScaler`**: Verilerin ortalamasını $0$, standart sapmasını $1$ yapacak şekilde ölçeklendirir (standartlaştırma). Lojistik Regresyon gibi gradyan inişi kullanan modellerin daha hızlı yakınsamasını sağlar ve katsayıların (ağırlıkların) birbirleriyle kıyaslanabilir olmasını kolaylaştırır.
*   **`PolynomialFeatures`**: Mevcut özniteliklerin (features) birbirleriyle çarpımlarını ve üslerini alarak yeni sentetik öznitelikler üretir (örneğin $x_1$ ve $x_2$ verildiyse; $x_1^2, x_2^2, x_1 x_2$ gibi). Doğrusal (lineer) modellerin, verideki doğrusal olmayan (non-linear) karmaşık sınırları ve ilişkileri öğrenebilmesini sağlar.
*   **`LogisticRegression`**: Scikit-Learn'ün doğrusal ikili sınıflandırma (veya `multi_class` ile çok sınıflı sınıflandırma) modelidir. L2 (Ridge) veya L1 (Lasso) düzenlileştirme yöntemlerini ve farklı optimizasyon çözücülerini (solver) destekler.
*   **`make_pipeline`**: Veri önişleme adımlarını (örneğin `StandardScaler`) ve modeli (`LogisticRegression`) tek bir ardışık düzen (pipeline) altında birleştirir. Bu sayede verinin eğitim ve test aşamalarında aynı önişleme adımlarından geçmesi garanti edilir ve veri sızıntısı (data leakage) engellenir.
*   **`train_test_split`**: Veri kümesini rastgele eğitim (train) ve test (test) alt kümelerine böler. `stratify=y` parametresi ile hedef sınıf oranlarının her iki kümede de korunmasını sağlar.
*   **`cross_val_score`**: K-Katlamalı Çapraz Doğrulama (K-Fold Cross-Validation) gerçekleştirerek modelin genelleştirme performansını veri setinin farklı bölümleri üzerinde test eder, daha güvenilir bir metrik ortalaması sunar.
*   **`confusion_matrix` (Karmaşıklık Matrisi)**: Modelin doğru ve yanlış sınıflandırma kararlarının (Doğru Pozitif, Doğru Negatif, Yanlış Pozitif, Yanlış Negatif) sayısal dağılımını gösterir.
*   **`classification_report`**: Sınıflandırma modelinin başarısını ölçen temel metrikleri (`Precision`, `Recall`, `F1-score` ve `Support`) sınıf bazında rapor halinde sunar.
*   **`roc_auc_score`**: ROC eğrisinin altında kalan alanı (AUC) hesaplar. Modelin sınıfları birbirinden ayırt edebilme yeteneğini (discrimination power) eşik değerinden bağımsız olarak ölçer. $1.0$ mükemmel sınıflandırmayı, $0.5$ ise rastgele tahmini temsil eder.

### PyTorch Bileşenleri

*   **`nn.Linear`**: Giriş özniteliklerine doğrusal bir dönüşüm uygular ($y = xA^T + b$). Lojistik regresyonun doğrusal kısmını temsil eden katsayılar (weights) ve sabit terimi (bias) barındırır.
*   **`nn.BCEWithLogitsLoss`**: İkili Çapraz Entropi (Binary Cross Entropy) kaybını hesaplar. İçerisinde otomatik olarak bir Sigmoid katmanı barındırdığı için sayısal olarak daha kararlıdır (`nn.BCELoss` ve manuel Sigmoid kullanımına kıyasla alt/üst taşmaları önler).
*   **`optim.SGD`**: Stokastik Gradyan İnişi (Stochastic Gradient Descent) optimizasyon algoritmasıdır. Model katsayılarını hesaplanan gradyanlar ve öğrenme oranı (learning rate) doğrultusunda günceller.